Mount ADLS Gen2 containers to Databricks DBFS
Using **Storage Account Access Key** (no Entra ID)

In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook: 00_mount_adls.py
# Purpose:  Mount ADLS Gen2 containers to Databricks DBFS
#           Using Storage Account Access Key (no Entra ID)
# ─────────────────────────────────────────────────────────────

# ── Read access key from Key Vault secret scope ──────────────
STORAGE_ACCOUNT = "walmartdatalake"
ACCESS_KEY      = dbutils.secrets.get(scope="walmart-kv-scope", key="adls-access-key")

# ── Config using access key ──────────────────────────────────
configs = {
    f"fs.azure.account.key.{STORAGE_ACCOUNT}.dfs.core.windows.net": ACCESS_KEY
}

# ── Mount function ───────────────────────────────────────────
def mount_container(container: str, mount_point: str):
    # Unmount if already mounted
    if any(mount.mountPoint == mount_point for mount in dbutils.fs.mounts()):
        dbutils.fs.unmount(mount_point)
        print(f"  Unmounted existing: {mount_point}")

    dbutils.fs.mount(
        source      = f"abfss://{container}@{STORAGE_ACCOUNT}.dfs.core.windows.net/",
        mount_point = mount_point,
        extra_configs = configs
    )
    print(f"  ✅ Mounted {container} → {mount_point}")

# ── Mount all three containers ───────────────────────────────
print("Mounting ADLS Gen2 containers...")
mount_container("bronze", "/mnt/bronze")
mount_container("silver", "/mnt/silver")
mount_container("gold",   "/mnt/gold")

print("\nAll mounts complete!")

# ── Verify mounts ────────────────────────────────────────────
print("\nVerifying Bronze contents:")
display(dbutils.fs.ls("/mnt/bronze"))


Mount ADLS Gen2 containers to Databricks DBFS
Using **Access Connector** (no keys, no Entra secrets)

In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook: 00_mount_adls.py
# Purpose:  Mount ADLS Gen2 containers to Databricks DBFS
#           Using Access Connector (no keys, no Entra secrets)
# ─────────────────────────────────────────────────────────────

STORAGE_ACCOUNT = "walmartdata"

# ── Config using Access Connector ────────────────────────────
# Replace with your Access Connector Resource ID
ACCESS_CONNECTOR_RESOURCE_ID = "/subscriptions/7729b227-1dd0-41ab-8e14-2364ce1540d9/resourceGroups/walmart-retail/providers/Microsoft.Databricks/accessConnectors/databricks-connector"

configs = {
    "fs.azure.account.auth.type": "CustomAccessToken",
    "fs.azure.account.custom.token.provider.class": "com.databricks.backend.daemon.dbutils.AzureAccessConnectorTokenProvider",
    "fs.azure.account.custom.token.provider.resourceId": ACCESS_CONNECTOR_RESOURCE_ID
}

# ── Mount function ───────────────────────────────────────────
def mount_container(container: str, mount_point: str):
    try:
        # Try listing the mount point to see if it exists
        dbutils.fs.ls(mount_point)
        dbutils.fs.unmount(mount_point)
        print(f"  Unmounted existing: {mount_point}")
    except Exception:
        # If it fails, the mount point doesn't exist yet
        pass

    # Perform mount
    dbutils.fs.mount(
        source      = f"abfss://{container}@{STORAGE_ACCOUNT}.dfs.core.windows.net/",
        mount_point = mount_point,
        extra_configs = configs
    )
    print(f"  ✅ Mounted {container} → {mount_point}")




# ── Mount all three containers ───────────────────────────────
print("Mounting ADLS Gen2 containers...")
mount_container("bronze", "/mnt/bronze")
mount_container("silver", "/mnt/silver")
mount_container("gold",   "/mnt/gold")

print("\nAll mounts complete!")

# ── Verify mounts ────────────────────────────────────────────
print("\nVerifying Bronze contents:")
display(dbutils.fs.ls("/mnt/bronze"))

**Unity Catalog** external locations

In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook: 00_verify_external_locations.py
# Purpose:  Sanity check that Unity Catalog external locations
#           are accessible before running the pipeline
# ─────────────────────────────────────────────────────────────

STORAGE_ACCOUNT = "walmartdata"

PATHS = {
    "bronze": f"abfss://bronze@{STORAGE_ACCOUNT}.dfs.core.windows.net/",
    "silver": f"abfss://silver@{STORAGE_ACCOUNT}.dfs.core.windows.net/",
    "gold":   f"abfss://gold@{STORAGE_ACCOUNT}.dfs.core.windows.net/",
}

print("Verifying Unity Catalog external locations...\n")

for layer, path in PATHS.items():
    try:
        dbutils.fs.ls(path)
        print(f"  ✅ {layer.upper()} accessible → {path}")
    except Exception as e:
        print(f"  ❌ {layer.upper()} FAILED → {path}\n     {e}")

print("\nAll checks complete — ready to run the pipeline!")